### Notebook 范围

### 目的
计算 FWD 和 INT/NOCOUPL skill，并生成最终 synthesis matrix。

### 科学问题
interactive O3 是否延迟 50 hPa final warming？是否改变 O3 pathway skill？哪些机制证据在多 case 中稳健？

### 方法
FWD 明确定义为 U60N50 < 7 m/s 并持续 10 天的第一天。O3 skill 同时给出 init-May30 RMSE、Mar-Apr RMSE 和 Mar-Apr minimum error。最终矩阵只显示数值和 NA，不再使用 √ 或 ?。

### 输出
outputs/figures/08_final_warming_skill 和 outputs/figures/99_synthesis。

### 解读
FWD 正差值表示 INT 比 NOCOUPL 更晚 final warming；RMSE 负差值表示 INT 误差更小。

### 注意
FWD 若在 hindcast 时段内未跨阈值则为 NA；skill 是相对于 BWCN same-year reference，不等同 Marina 复现。


### 导入与公共路径

### 目的
加载 Extention_analysis 公共函数。

### 科学问题
所有 notebook 是否共享相同路径、变量定义和 sign convention？

### 方法
导入 hindcast_extension_utils.py。

### 输出
无图。

### 解读
路径正确即可继续。

### 注意
所有输出都写入 outputs 子目录。


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 计算 FWD 和 skill

### 目的
对 paired INT/NOCOUPL cases 计算 FWD distribution 和 O3 skill metrics。

### 科学问题
INT 是否系统性延迟 50 hPa final warming？O3 skill 改善还是只是 pathway 改变？

### 方法
FWD 基于 U60N50；O3_RMSE_init_May30、O3_RMSE_MarApr、O3_min_error_MarApr 均相对同年 BWCN reference。

### 输出
08_FWD_distributions_INT_vs_NOCOUPL_v2、08_INT_vs_NOCOUPL_skill_difference_heatmap_v2。

### 解读
正 ΔFWD 表示 INT 延迟 final warming；负 ΔRMSE 表示 INT 更接近 reference。

### 注意
sample size 小，且 paired comparison 只包含 small perturbation INT/NOCOUPL。


In [ ]:
fig_dir = figure_dir("08_final_warming_skill")
tab_dir = table_dir("08_final_warming_skill")
inv = discover_hindcast_cases(); pairs = paired_int_nocoupl_cases(inv)
fwd_rows = []
skill_rows = []
for _, pair in pairs.iterrows():
    for config, case in [("INT", pair["INT_case"]), ("NOCOUPL", pair["NOCOUPL_case"] )]:
        year = parse_case_name(case)["year"]
        u50, _ = compute_u60(case, 50)
        fwd = compute_fwd_from_u60n50(u50)
        fwd["case"] = case; fwd["config"] = config; fwd["year"] = pair["year"]; fwd["init_month"] = pair["init_month"]
        fwd_rows.append(fwd)
        o3, o3_date = load_hindcast_o3(case)
        ref, ref_date = load_bwcn_reference_o3(year)
        rmse_full = compute_o3_rmse(o3, ref, *target_window_for_case(case)).rename(columns={"O3_RMSE": "O3_RMSE_init_May30"})
        rmse_ma = compute_o3_rmse(o3, ref, (3, 1), (4, 30)).rename(columns={"O3_RMSE": "O3_RMSE_MarApr"})
        ma_min = o3_ma_min(o3, o3_date).rename(columns={"O3_MA_min": "O3_MA_min_member"})
        ref_min = o3_ma_min(ref, ref_date).rename(columns={"O3_MA_min": "O3_MA_min_reference"})
        ref_val = float(ref_min["O3_MA_min_reference"].iloc[0]) if not ref_min.empty else np.nan
        skill = rmse_full.merge(rmse_ma, on="member", how="outer").merge(ma_min, on="member", how="outer")
        skill["O3_min_error_MarApr"] = skill["O3_MA_min_member"] - ref_val
        skill["case"] = case; skill["config"] = config; skill["year"] = pair["year"]; skill["init_month"] = pair["init_month"]
        skill_rows.append(skill)
fwd_df = pd.concat(fwd_rows, ignore_index=True) if fwd_rows else pd.DataFrame()
skill_df = pd.concat(skill_rows, ignore_index=True) if skill_rows else pd.DataFrame()
fwd_csv = tab_dir / "08_FWD_INT_NOCOUPL_summary.csv"; skill_csv = tab_dir / "08_INT_vs_NOCOUPL_skill_metrics.csv"
fwd_df.to_csv(fwd_csv, index=False); skill_df.to_csv(skill_csv, index=False)
fwd_df.to_csv(TAB_DIR / "08_FWD_INT_NOCOUPL_summary.csv", index=False); skill_df.to_csv(TAB_DIR / "08_INT_vs_NOCOUPL_skill_metrics.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
if fwd_df.empty:
    for ax in axes: ax.axis("off")
else:
    labels = []
    for i, ((year, init), sub) in enumerate(fwd_df.groupby(["year", "init_month"])):
        labels.append(f"{year}-{init}")
        for cfg, off, color in [("INT", -0.14, "tab:blue"), ("NOCOUPL", 0.14, "tab:orange")]:
            vals = sub.loc[sub["config"] == cfg, "FWD_DOY"].dropna()
            axes[0].scatter(np.full(len(vals), i + off), vals, s=28, alpha=0.7, color=color, label=cfg if i == 0 else None)
        piv = sub.pivot_table(index="member", columns="config", values="FWD_DOY")
        if set(["INT", "NOCOUPL"]).issubset(piv.columns):
            delta = piv["INT"] - piv["NOCOUPL"]
            axes[1].scatter(np.full(len(delta.dropna()), i), delta.dropna(), color="tab:purple", s=30)
            axes[1].plot([i-0.22, i+0.22], [delta.mean(), delta.mean()], color="black", lw=2)
    axes[0].set_xticks(range(len(labels)), labels, rotation=45, ha="right"); axes[0].set_ylabel("FWD DOY at 50 hPa"); axes[0].legend(); axes[0].set_title("Final warming date distribution\nU60N50 < 7 m/s for ≥10 days")
    axes[1].axhline(0, color="0.3", lw=0.8); axes[1].set_xticks(range(len(labels)), labels, rotation=45, ha="right"); axes[1].set_ylabel("INT - NOCOUPL FWD (days)"); axes[1].set_title("Positive = INT delays 50 hPa final warming")
savefig(fig, "08_FWD_distributions_INT_vs_NOCOUPL_v2", fig_dir=fig_dir, notebook="08_final_warming_and_skill.ipynb", scientific_question="interactive O3 是否延迟 50 hPa final warming？", variables_windows="FWD from U60N50 < 7 m/s sustained 10 days", interpretation="正 INT-NOCOUPL 表示 INT final warming 更晚。", caveat="若 hindcast 时段内未跨阈值则 FWD=NA。", csv_table=fwd_csv)
plt.show()

skill_delta_rows = []
for (year, init), sub in skill_df.groupby(["year", "init_month"]) if not skill_df.empty else []:
    for metric in ["O3_RMSE_init_May30", "O3_RMSE_MarApr", "O3_min_error_MarApr"]:
        piv = sub.pivot_table(index="member", columns="config", values=metric)
        if set(["INT", "NOCOUPL"]).issubset(piv.columns):
            delta = piv["INT"] - piv["NOCOUPL"]
            skill_delta_rows.append({"year": year, "init_month": init, "metric": metric, "mean_INT_minus_NOCOUPL": float(delta.mean()), "n": int(delta.dropna().shape[0])})
skill_delta = pd.DataFrame(skill_delta_rows)
skill_delta_csv = tab_dir / "08_INT_vs_NOCOUPL_skill_difference_heatmap.csv"
skill_delta.to_csv(skill_delta_csv, index=False)
skill_delta.to_csv(TAB_DIR / "08_INT_vs_NOCOUPL_skill_difference_heatmap.csv", index=False)
mat = skill_delta.pivot(index=["year", "init_month"], columns="metric", values="mean_INT_minus_NOCOUPL") if not skill_delta.empty else pd.DataFrame()
fig, ax = plt.subplots(figsize=(8.8, max(3.6, 0.4 * len(mat) + 2)), constrained_layout=True)
if mat.empty:
    ax.axis("off"); ax.text(0.5, 0.5, "No skill deltas", ha="center")
else:
    vmax = np.nanmax(np.abs(mat.values)); vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1
    im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(mat.shape[1]), mat.columns, rotation=35, ha="right")
    ax.set_yticks(range(mat.shape[0]), [f"{a}-{b}" for a, b in mat.index])
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.iloc[i, j]
            ax.text(j, i, "NA" if not np.isfinite(val) else f"{val:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title("INT - NOCOUPL O3 skill differences\nnegative RMSE delta = INT closer to BWCN reference")
    fig.colorbar(im, ax=ax, label="mean INT - NOCOUPL")
savefig(fig, "08_INT_vs_NOCOUPL_skill_difference_heatmap_v2", fig_dir=fig_dir, notebook="08_final_warming_and_skill.ipynb", scientific_question="interactive O3 是否改变 O3 pathway skill？", variables_windows="O3_RMSE_init_May30, O3_RMSE_MarApr, O3_min_error_MarApr relative to BWCN same-year reference", interpretation="负 RMSE delta 表示 INT 误差更小；min error 符号表示最低臭氧偏差方向。", caveat="skill 不等于物理反馈强弱；需结合 evolution/FWD。", csv_table=skill_delta_csv)
plt.show()


### 最终 synthesis matrix

### 目的
整合 01/03/04/05/07/08 的诊断成一个数值矩阵。

### 科学问题
机制链条哪些环节跨 case 稳健，哪些只适用于 0008-01 或 paired feedback？

### 方法
矩阵只使用数值和 NA，不使用 √、? 或 /。列包括 event severity、EP100-O3、W1+W2 dominance、spread lead、Z300 source、FWD delay、skill delta。

### 输出
final_mechanism_robustness_summary_matrix_v2.png/pdf 和 CSV。

### 解读
数值越强越支持相应机制；NA 表示不适用或没有数据。

### 注意
这是索引图，不应替代各 notebook 的具体图和表。


In [ ]:
fig_dir = figure_dir("99_synthesis")
tab_dir = table_dir("99_synthesis")
inv = discover_hindcast_cases()
matrix = inv[["case", "year", "init_month", "config", "perturbation"]].copy()
for col in ["MA_min_O3_DU", "EP100_O3_link_R", "W1W2_dominance_R", "spread_lead_O3_minus_dyn_days", "Z300_projection_EP100anom_R", "Z300_wave2_EP100anom_R", "INT_minus_NOCOUPL_FWD_days", "INT_minus_NOCOUPL_O3_RMSE_MA"]:
    matrix[col] = np.nan
sev_path = TAB_DIR / "01_reference_O3_severity_selected_years.csv"
if sev_path.exists():
    sev = pd.read_csv(sev_path, dtype={"year": str})
    sev["year"] = sev["year"].astype(str).str.zfill(4)
    sev_map = sev.set_index("year")["MA_min_O3_DU"].to_dict()
    matrix["MA_min_O3_DU"] = matrix["year"].astype(str).str.zfill(4).map(sev_map)
cor03 = TAB_DIR / "03_EP100_O3_U_FWD_correlations_all_cases.csv"
if cor03.exists():
    c03 = pd.read_csv(cor03, dtype={"case": str})
    for _, r in c03.iterrows():
        metric = r.get("metric", r.get("relationship", ""))
        if metric == "wave1_plus_wave2 vs O3_RMSE":
            matrix.loc[matrix["case"] == r["case"], "EP100_O3_link_R"] = r["R"]
        if metric == "all_waves vs W1+W2":
            matrix.loc[matrix["case"] == r["case"], "W1W2_dominance_R"] = r["R"]
spread_path = TAB_DIR / "04_spread_onset_timing_summary_all_cases.csv"
if spread_path.exists():
    sp = pd.read_csv(spread_path, dtype={"case": str})
    piv = sp.pivot_table(index="case", columns="variable", values="onset_lead_day", aggfunc="first")
    for case in piv.index:
        dyn = np.nanmin([piv.loc[case].get("EP100_W1W2", np.nan), piv.loc[case].get("U60N10", np.nan), piv.loc[case].get("U60N50", np.nan)])
        o3 = piv.loc[case].get("O3", np.nan)
        matrix.loc[matrix["case"] == case, "spread_lead_O3_minus_dyn_days"] = o3 - dyn if np.isfinite(o3) and np.isfinite(dyn) else np.nan
zcor_path = TAB_DIR / "05_Z300_source_metric_correlations_all_cases.csv"
if zcor_path.exists():
    zc = pd.read_csv(zcor_path, dtype={"case": str})
    # Use the average Jan/Feb relationship by case for synthesis.
    for case, sub in zc.groupby("case"):
        a = sub[sub["relationship"] == "Z300_stationary_projection_to_B2000 vs EP100_wave1_plus_wave2_anom_ref"]["R"]
        b = sub[sub["relationship"] == "Z300_wave2_amplitude_m vs EP100_wave2_anom_ref"]["R"]
        if len(a): matrix.loc[matrix["case"] == case, "Z300_projection_EP100anom_R"] = float(a.mean())
        if len(b): matrix.loc[matrix["case"] == case, "Z300_wave2_EP100anom_R"] = float(b.mean())
fwd_path = TAB_DIR / "08_FWD_INT_NOCOUPL_summary.csv"
if fwd_path.exists():
    fwd = pd.read_csv(fwd_path, dtype={"member": str, "year": str, "init_month": str})
    for (year, init), sub in fwd.groupby(["year", "init_month"]):
        piv = sub.pivot_table(index="member", columns="config", values="FWD_DOY")
        if set(["INT", "NOCOUPL"]).issubset(piv.columns):
            delta = float((piv["INT"] - piv["NOCOUPL"]).mean())
            mask = (matrix["year"].astype(str).str.zfill(4) == str(year).zfill(4)) & (matrix["init_month"].astype(str).str.zfill(2) == str(init).zfill(2))
            matrix.loc[mask, "INT_minus_NOCOUPL_FWD_days"] = delta
skill_path = TAB_DIR / "08_INT_vs_NOCOUPL_skill_difference_heatmap.csv"
if skill_path.exists():
    sk = pd.read_csv(skill_path, dtype={"year": str, "init_month": str})
    sub = sk[sk["metric"] == "O3_RMSE_MarApr"]
    for _, r in sub.iterrows():
        mask = (matrix["year"].astype(str).str.zfill(4) == str(r["year"]).zfill(4)) & (matrix["init_month"].astype(str).str.zfill(2) == str(r["init_month"]).zfill(2))
        matrix.loc[mask, "INT_minus_NOCOUPL_O3_RMSE_MA"] = r["mean_INT_minus_NOCOUPL"]
out_csv = tab_dir / "final_mechanism_robustness_summary_matrix_v2.csv"
matrix.to_csv(out_csv, index=False)
matrix.to_csv(TAB_DIR / "final_mechanism_robustness_summary_matrix.csv", index=False)
plot_cols = ["MA_min_O3_DU", "EP100_O3_link_R", "W1W2_dominance_R", "spread_lead_O3_minus_dyn_days", "Z300_projection_EP100anom_R", "Z300_wave2_EP100anom_R", "INT_minus_NOCOUPL_FWD_days", "INT_minus_NOCOUPL_O3_RMSE_MA"]
vals = matrix[plot_cols].astype(float)
scaled = vals.copy()
for col in plot_cols:
    v = vals[col].values.astype(float)
    finite = np.isfinite(v)
    if finite.any():
        lo, hi = np.nanpercentile(v[finite], [5, 95])
        if hi > lo:
            scaled[col] = np.clip((v - lo) / (hi - lo) * 2 - 1, -1, 1)
        else:
            scaled[col] = 0
fig, ax = plt.subplots(figsize=(14, max(5, 0.34 * len(matrix) + 2)), constrained_layout=True)
im = ax.imshow(scaled[plot_cols].values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(plot_cols)), plot_cols, rotation=45, ha="right")
ax.set_yticks(range(len(matrix)), matrix["case"])
for i in range(len(matrix)):
    for j, col in enumerate(plot_cols):
        val = vals.iloc[i, j]
        ax.text(j, i, "NA" if not np.isfinite(val) else f"{val:.2g}", ha="center", va="center", fontsize=7)
ax.set_title("Mechanism robustness summary matrix (numeric values; NA = not applicable/missing)\nNo check marks: read each column by its physical sign and units")
fig.colorbar(im, ax=ax, label="column-wise scaled value for display only")
savefig(fig, "final_mechanism_robustness_summary_matrix_v2", fig_dir=fig_dir, notebook="08_final_warming_and_skill.ipynb", scientific_question="哪些机制证据跨 hindcast cases 稳健？", variables_windows="Compiled O3 severity, EP100, spread timing, Z300 Jan/Feb source, INT/NOCOUPL FWD and O3 skill", interpretation="图中文字是实际数值；颜色只是每列缩放后的显示辅助。NA 表示不适用或缺数据。", caveat="这是索引矩阵，必须回看各分图解释符号和窗口。", csv_table=out_csv)
plt.show()
write_figure_guide()
with (OUT_ROOT / "EXTENTION_ANALYSIS_SYNTHESIS.md").open("w") as f:
    f.write("""# Extention Analysis Synthesis\n\n0008-01 provides the detailed source mechanism:\nZ300 stationary-wave geometry / wave-2 amplitude -> constructive/destructive interference -> EP100 W1+W2 anomaly -> U60N10/U60N50 vortex pathway spread -> later O3 RMSE.\n\nThis revised workflow separates three interpretations:\n\n1. Jan/Feb source diagnosis: use Z300 stationary-wave metrics and EP100 reference anomalies only where the case actually contains Jan/Feb data.\n2. Post-initialization wave forcing: use primary/lead windows for all cases, especially later initialized cases.\n3. INT-vs-NOCOUPL feedback: use paired small-perturbation cases to test ozone feedback on O3, vortex wind, EP100, FWD, and skill.\n\nLater-initialized cases should not be interpreted as long-lead precursor tests. They are better suited for diagnosing early post-initialization wave forcing, vortex persistence, final warming, and feedback differences.\n""")
